# One audit per ontology

For every ontology, this notebook writes a short audit in `audits/<ACRONYM>.md`: its scores, the O'FAIRe
questions where it loses points, the cheatsheet attributes that are empty, and who can fill them in.
It also writes `audits/index.md` (all ontologies, best first) and `data/actions_<RUN_DATE>.csv`, the base
for a curation strategy.

- **I.** Loading what the previous notebooks produced
- **II.** Empty attributes and the points they cost
- **III.** Writing the audits
- **IV.** A first look at the actions

In [1]:
# Imports
import json

import pandas as pd

from config import *

# I. Loading what the previous notebooks produced

In [2]:
catalogue = {o["acronym"]: o for o in json.load(open(RAW / "catalogue.json", encoding="utf-8"))}
submissions = json.load(open(RAW / "submissions.json", encoding="utf-8"))
ofaire = json.load(open(RAW / "ofaire.json", encoding="utf-8"))
scores = pd.read_csv(DATA / f"ofaire_scores_{RUN_DATE}.csv").set_index("acronym")
questions = pd.read_csv(DATA / f"ofaire_questions_{RUN_DATE}.csv").set_index("question")
guideline = pd.read_csv(MAPPING / "guideline_profile.csv", dtype={"id": str}).fillna("")
links = pd.read_csv(MAPPING / "ofaire_vs_guideline.csv").fillna("")

reachable = questions[questions.evaluable & questions.reachable].index.tolist()
questions_of = links[links.cheatsheet_entry != ""].groupby("cheatsheet_entry").question.apply(set).to_dict()

# We keep O'FAIRe's own explanation of each score, to quote it in the audits
explanation = {}
for acronym, result in ofaire.items():
    for principle in ("Findable", "Accessible", "Interoperable", "Reusable"):
        for block in result[principle].values():
            if isinstance(block, dict):
                for qid, q in block.get("results", {}).items():
                    explanation[(acronym, qid)] = (q.get("explanation") or "").strip()
print(f"{len(scores)} ontologies, {len(reachable)} reachable questions, {len(guideline)} cheatsheet attributes")

92 ontologies, 49 reachable questions, 99 cheatsheet attributes


# II. Empty attributes and the points they cost

An attribute is **empty** when its value is null, `""`, `[]` or `{}`; an agent or a contact whose record
has nothing but its identifier counts as empty too. When a question loses *L* points and reads *n* empty
attributes, each of them receives a share of *L / n*. This share is a way to rank the actions, not a
promise of points, because O'FAIRe has its own rule for each question. For example, I1Q5 needs
*Has format* and *Is format of* together, one attribute is enough for I2Q6, and F1Q2 gives its 11 points
only to a valid DOI.

Each empty attribute gets a **priority**: 1 = Mandatory; 2 = Recommended and costing O'FAIRe points;
3 = Recommended only; 4 = Optional but costing O'FAIRe points. Optional attributes that cost nothing are
not actions.

In [3]:
def filled(value):
    if value is None:
        return False
    if isinstance(value, bool):
        return True
    if isinstance(value, str):
        return value.strip() != ""
    if isinstance(value, list):
        return any(filled(v) for v in value)
    if isinstance(value, dict):
        return any(filled(v) for k, v in value.items() if k not in ("id", "@id", "@type", "@context"))
    return True


def value_of(acronym, attribute):
    record = catalogue[acronym] if attribute.portal_object == "ontology" else submissions.get(acronym, {})
    return record.get(attribute.api_key)


actions = []
lost_points = {}
for acronym, s in scores.iterrows():
    kind = s.kind
    lost = {q: questions.loc[q, "max_credits"] - s[q] for q in reachable
            if pd.notna(s[q]) and s[q] < questions.loc[q, "max_credits"]}
    lost_points[acronym] = lost
    empty = guideline[[not filled(value_of(acronym, a)) for a in guideline.itertuples()]]
    if kind != "view":                                  # only a view can be "a view of" something
        empty = empty[empty.entry != "This ontology is a view of"]
    if submissions.get(acronym, {}).get("status") != "retired":   # only a retired ontology "was invalidated"
        empty = empty[empty.entry != "Was invalidated by"]
    n_empty = {q: sum(q in questions_of.get(e, set()) for e in empty.entry) for q in lost}
    for a in empty.itertuples():
        qs = sorted(q for q in questions_of.get(a.entry, set()) if q in lost)
        share = round(sum(lost[q] / n_empty[q] for q in qs), 1)
        priority = {"Mandatory": 1, "Recommended": 2 if qs else 3}.get(a.usage, 4 if qs else None)
        if priority:
            actions.append({"acronym": acronym, "kind": kind, "priority": priority, "who_fixes": a.who_fixes,
                            "cheatsheet_id": a.id, "entry": a.entry, "usage": a.usage,
                            "portal_attribute": a.portal_attribute, "questions": " ".join(qs),
                            "points_share": share})
actions = pd.DataFrame(actions)
actions.to_csv(DATA / f"actions_{RUN_DATE}.csv", index=False)
print(f"{len(actions)} actions saved to {(DATA / f'actions_{RUN_DATE}.csv').relative_to(REPO)}")
actions.head()

5552 actions saved to earthportal_sa_audit/data/actions_2026-10-08.csv


,acronym,kind,priority,who_fixes,cheatsheet_id,entry,usage,portal_attribute,questions,points_share
0,OSO,artefact,2,author,6.2,Is format of,Recommended,dct:isFormatOf,I1Q5,4.0
1,OSO,artefact,3,author,8.6,Award,Recommended,schema:award,,0.0
2,OSO,artefact,3,author,10.4,Similar to,Recommended,door:similarTo,,0.0
3,OSO,artefact,3,author,10.10,Disparate modelling with,Recommended,door:hasDisparateModelling,,0.0
4,OSO,artefact,3,author,10.15,Used by,Recommended,voaf:usedBy,,0.0


# III. Writing the audits

In [4]:
def md(text):
    return str(text).replace("|", "\\|").replace("\n", " ").strip()

rank = scores[scores.kind == "artefact"].reachable_score_pct.rank(ascending=False, method="min")
n_artefacts = int((scores.kind == "artefact").sum())
WHO = {"author": "By the author (in the file or the submission form)",
       "curator": "By the curator (EarthPortal team)",
       "portal": "By the portal (generated values: check that processing succeeded)"}

for acronym, s in scores.iterrows():
    onto, mine = catalogue[acronym], actions[actions.acronym == acronym].sort_values(["priority", "points_share"], ascending=[True, False])
    lines = [f"# {acronym}: {md(onto['name'])}", "",
             f"Kind: **{s.kind}** · Snapshot: {RUN_DATE} · Portal page: https://earthportal.eu/ontologies/{acronym}", "",
             "| O'FAIRe score | Points | Reachable score | Rank |", "|---|---|---|---|",
             f"| {s.normalized_score} / 100 | {s.total_points:g} / 478 | {s.reachable_score_pct} % of {s.reachable_max} | "
             + (f"{int(rank[acronym])} of {n_artefacts} artefacts |" if s.kind == "artefact" else f"not ranked ({s.kind}) |"),
             "", "## Where O'FAIRe points are lost", ""]
    lost = sorted(lost_points[acronym].items(), key=lambda item: -item[1])
    if lost:
        lines += ["| Question | Points | Lost | O'FAIRe says |", "|---|---|---|---|"]
        lines += [f"| {q} | {s[q]:g} / {questions.loc[q, 'max_credits']:g} | {l:g} | {md(explanation.get((acronym, q), ''))} |" for q, l in lost]
    else:
        lines.append("No points lost on the reachable questions.")
    lines += ["", "## Suggested actions", "", "Priority 1: Mandatory. Priority 2: Recommended and costing O'FAIRe points. The share is the points lost "
              "on the questions in brackets, divided among the empty attributes these questions read.", ""]
    for who, title in WHO.items():
        todo = mine[(mine.who_fixes == who) & (mine.priority <= 2)]
        if len(todo):
            lines += [f"### {title}", ""]
            lines += [f"- [P{a.priority}] Fill **{a.entry}** (cheatsheet {a.cheatsheet_id}, {a.usage}, `{a.portal_attribute}`)"
                      + (f"; share of the O'FAIRe points lost: {a.points_share:g} ({a.questions})" if a.questions else "")
                      for a in todo.itertuples()]
            lines.append("")
    later = mine[mine.priority >= 3].entry.tolist()
    if later:
        lines.append(f"Later (priorities 3 and 4): {', '.join(later)}.")
    (AUDITS / f"{acronym}.md").write_text("\n".join(lines) + "\n", encoding="utf-8")

# The index: all ontologies, best reachable score first
p1 = actions[actions.priority == 1].groupby("acronym").size()
p2 = actions[actions.priority == 2].groupby("acronym").size()
index = ["# EarthPortal audits", "", f"Snapshot of {RUN_DATE}. *P1*: empty Mandatory attributes; "
         "*P2*: empty Recommended attributes that cost O'FAIRe points.", ""]
for kind, title in (("artefact", "Artefacts"), ("view", "Views"), ("metadata-only", "Records without content")):
    part = scores[scores.kind == kind].sort_values("reachable_score_pct", ascending=False)
    index += [f"## {title} ({len(part)})", "", "| Acronym | Name | O'FAIRe | Reachable | P1 | P2 |", "|---|---|---|---|---|---|"]
    index += [f"| [{a}]({a}.md) | {md(r['name'])} | {r.normalized_score} | {r.reachable_score_pct} % | {p1.get(a, 0)} | {p2.get(a, 0)} |"
              for a, r in part.iterrows()]
    index.append("")
(AUDITS / "index.md").write_text("\n".join(index), encoding="utf-8")
print(f"{len(scores)} audits and the index written to {AUDITS.relative_to(REPO)}/")

92 audits and the index written to earthportal_sa_audit/audits/


In [5]:
# We look at one audit, the best artefact
print((AUDITS / f"{scores[scores.kind == 'artefact'].reachable_score_pct.idxmax()}.md").read_text(encoding="utf-8"))

# OSO: Observatories of the Seas Ontology

Kind: **artefact** · Snapshot: 2026-10-08 · Portal page: https://earthportal.eu/ontologies/OSO

| O'FAIRe score | Points | Reachable score | Rank |
|---|---|---|---|
| 76 / 100 | 364.36 / 478 | 94.4 % of 386 | 1 of 68 artefacts |

## Where O'FAIRe points are lost

| Question | Points | Lost | O'FAIRe says |
|---|---|---|---|
| F4Q2 | 2 / 10 | 8 | The ontology is registered in 1 ontology repository. |
| A1Q3 | 18 / 24 | 6 | Ontology and its metadata support 6 negotiation formats. (Ontology accept the following formats: [text/html], Ontology metadata accept the following formats: [application/json, application/xml, text/html, text/plain, application/rdf+xml]) |
| I1Q5 | 0 / 4 | 4 | No information is provided about the availability of the ontology in other formats. |
| R1.2Q2 | 4 / 6 | 2 | The ontology provides 2 properties to inform on its general provenance. |
| I2Q7 | 3.36 / 5 | 1.64 | The ontology declares reusing standards metadata vocabular

# IV. A first look at the actions

In [6]:
# Artefacts only, priorities 1 and 2: who has the most to do?
top = actions[(actions.kind == "artefact") & (actions.priority <= 2)]
pd.crosstab(top.who_fixes, top.priority, margins=True)

priority,1,2,All
who_fixes,,,
author,89,1819,1908
curator,20,61,81
portal,37,0,37
All,146,1880,2026


In [7]:
# The Mandatory attributes most often empty
actions[(actions.kind == "artefact") & (actions.priority == 1)].entry.value_counts()

entry
Modification date            27
Location                     24
Creator                      23
Categories                   20
License                      15
CSV dump                     12
Download URL                  9
URI lookup Endpoint           9
Free-text search endpoint     7
Name: count, dtype: int64

**Conclusion**: most of the work falls on the authors, and a small part on the EarthPortal team. The
`who_fixes` column is a proposal to discuss with the team before contacting anyone.